In [1]:
"""V15 — TCN → BiLSTM → GRU → Attention + UNet Decoder"""
import os, gc, time, math, random
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim.swa_utils import AveragedModel
torch.backends.cudnn.benchmark = True

DATA_ROOT = "/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2"
TRAIN_DIR = os.path.join(DATA_ROOT, "raw")
TEST_DIR  = os.path.join(DATA_ROOT, "test_in")
OUT_DIR   = "/kaggle/working"
MONTHS = ["APRIL_16", "JULY_16", "OCT_16", "DEC_16"]
LIVE = ["cpm25","pblh","psfc","q2","rain","swdown","t2","u10","v10"]
LOG1P = {"cpm25", "rain"}; TARGET = "cpm25"
H, W = 140, 124
LOOKBACK, HORIZON = 10, 16
VAL_HOURS, VAL_GAP = 72, 16
SEED = 42
BATCH    = 4
BASE_CH  = 48
LR       = 5e-4
N_EPOCHS = 30      # ← change to 35 for full run
SWA_START= 25
DROPOUT  = 0.20
PATIENCE = 15
EP_BOOST = 4.0
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP  = True
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print(f"V15 TCN-BiLSTM-GRU-Attn | BS={BATCH} ch={BASE_CH} drop={DROPOUT} | {DEVICE}")

V15 TCN-BiLSTM-GRU-Attn | BS=4 ch=48 drop=0.2 | cuda


In [2]:
# ═══ Lucifer 13ch + 2 extra features ═══
CPM25_CH = 0
class ZNorm:
    def __init__(self): self.stats={}; self.dstats={}
    def fit(self, md):
        for f in LIVE:
            lg = f in LOG1P; v=[]
            for m in md:
                a=md[m][f].ravel()
                if lg: a=np.log1p(np.clip(a,0,None))
                v.append(a)
            v=np.concatenate(v); self.stats[f]=(float(v.mean()),float(max(v.std(),1e-8)),lg)
    def tr(self, a, f):
        m,s,lg=self.stats[f]
        if lg: a=np.log1p(np.clip(a,0,None))
        return ((a-m)/s).astype(np.float32)
    def fd(self, n, v): self.dstats[n]=(float(v.mean()),float(max(v.std(),1e-8)))
    def td(self, a, n): m,s=self.dstats[n]; return ((a-m)/s).astype(np.float32)
    def inv(self, a):
        m,s,lg=self.stats[TARGET]; a=a*s+m
        if lg: a=np.expm1(a)
        return a

def eng_month(md, norm):
    chs=[]
    for f in LIVE:
        if f in ('u10','v10'): continue
        chs.append(norm.tr(md[f].copy(), f))
    cn = norm.tr(md['cpm25'].copy(), 'cpm25')
    d=np.zeros_like(cn); d[1:]=cn[1:]-cn[:-1]; chs.append(d)
    u,v = md['u10'], md['v10']
    ws=np.sqrt(u**2+v**2); th=np.arctan2(v,u)
    for a,n in [(ws,'ws10'),(np.sin(th),'wsin'),(np.cos(th),'wcos')]:
        if n not in norm.dstats: norm.fd(n, a.ravel())
        chs.append(norm.td(a, n))
    chs.append((md['pblh']<300).astype(np.float32))
    vt=md['pblh']*ws
    if 'vent' not in norm.dstats: norm.fd('vent', vt.ravel())
    chs.append(norm.td(vt, 'vent'))
    # Extra: rolling 3h anomaly
    rm3=np.zeros_like(cn)
    for t in range(len(cn)): rm3[t]=cn[max(0,t-2):t+1].mean(0)
    chs.append(cn - rm3)
    # Extra: t2*q2 interaction
    t2n=norm.tr(md['t2'].copy(),'t2'); q2n=norm.tr(md['q2'].copy(),'q2')
    chs.append(t2n * q2n)
    return np.stack(chs, axis=1), cn

def eng_test(tf, norm):
    chs=[]
    for f in LIVE:
        if f in ('u10','v10'): continue
        chs.append(norm.tr(tf[f].copy(), f))
    cn=norm.tr(tf['cpm25'].copy(),'cpm25')
    d=np.zeros_like(cn); d[:,1:]=cn[:,1:]-cn[:,:-1]; chs.append(d)
    u,v=tf['u10'],tf['v10']; ws=np.sqrt(u**2+v**2); th=np.arctan2(v,u)
    for a,n in [(ws,'ws10'),(np.sin(th),'wsin'),(np.cos(th),'wcos')]:
        chs.append(norm.td(a, n))
    chs.append((tf['pblh']<300).astype(np.float32))
    chs.append(norm.td(tf['pblh']*ws, 'vent'))
    rm3=np.zeros_like(cn)
    for t in range(cn.shape[1]): rm3[:,t]=cn[:,max(0,t-2):t+1].mean(1)
    chs.append(cn-rm3)
    t2n=norm.tr(tf['t2'].copy(),'t2'); q2n=norm.tr(tf['q2'].copy(),'q2')
    chs.append(t2n*q2n)
    return np.stack(chs, axis=2)

print('Loading...')
raw={}
for m in MONTHS:
    raw[m]={}
    for f in LIVE:
        a=np.load(os.path.join(TRAIN_DIR,m,f'{f}.npy')).astype(np.float32)
        if a.ndim==4: a=a.squeeze(0)
        raw[m][f]=a
    print(f'  {m}: T={raw[m][TARGET].shape[0]}')
norm=ZNorm(); norm.fit(raw)
mengs,mtgts={},{}
for i,m in enumerate(MONTHS):
    e,t=eng_month(raw[m],norm); mengs[i]=e; mtgts[i]=t
    print(f'  {m}: {e.shape}')
IN_CH=mengs[0].shape[1]; print(f'Channels: {IN_CH} ✅')
del raw; gc.collect()

Loading...
  APRIL_16: T=715
  JULY_16: T=739
  OCT_16: T=739
  DEC_16: T=739
  APRIL_16: (715, 15, 140, 124)
  JULY_16: (739, 15, 140, 124)
  OCT_16: (739, 15, 140, 124)
  DEC_16: (739, 15, 140, 124)
Channels: 15 ✅


433

In [3]:
# ═══ Episode Detection + Dataset ═══
def ep_mask(cn, w=24, sig=1.5):
    T=cn.shape[0]; ep=np.zeros_like(cn,dtype=np.float32)
    for t in range(w,T):
        b=cn[t-w:t]; ep[t]=(np.abs(cn[t]-b.mean(0))>sig*(b.std(0)+1e-8)).astype(np.float32)
    return ep
WIN=LOOKBACK+HORIZON
mep={}; train_idx=[]; val_idx=[]; ep_dens=[]
for mi in range(4):
    mep[mi]=ep_mask(mtgts[mi])
    T=mengs[mi].shape[0]; vs=T-VAL_HOURS; te=vs-VAL_GAP
    for t in range(te-WIN+1):
        train_idx.append((mi,t))
        ep_dens.append(mep[mi][t+LOOKBACK:t+WIN].sum()/(HORIZON*H*W))
    for t in range(max(0,vs-LOOKBACK), T-WIN+1):
        val_idx.append((mi,t))
ep_dens=np.array(ep_dens)
tw=np.where(ep_dens>np.median(ep_dens), EP_BOOST, 1.0)
tw[ep_dens>np.percentile(ep_dens,90)]=EP_BOOST*2

class DS(Dataset):
    def __init__(s,idx,aug=False): s.idx=idx; s.aug=aug
    def __len__(s): return len(s.idx)
    def __getitem__(s,i):
        mi,t=s.idx[i]; x=mengs[mi][t:t+LOOKBACK].copy(); y=mtgts[mi][t+LOOKBACK:t+WIN].copy()
        if s.aug:
            if random.random()<.5: x=x[:,:,:,::-1].copy(); y=y[:,:,::-1].copy()
            if random.random()<.5: x=x[:,:,::-1,:].copy(); y=y[:,::-1,:].copy()
        return torch.from_numpy(x), torch.from_numpy(y)

tds=DS(train_idx,True); vds=DS(val_idx)
smp=WeightedRandomSampler(tw.tolist(),len(train_idx),replacement=True)
tl=DataLoader(tds,BATCH,sampler=smp,num_workers=2,pin_memory=True,drop_last=True)
vl=DataLoader(vds,BATCH,shuffle=False,num_workers=2,pin_memory=True)
print(f'Train:{len(tl)} Val:{len(vl)} ✅')

Train:620 Val:57 ✅


In [4]:
# ═══ Model: TCN → BiLSTM → GRU → Attention + UNet ═══
class CB(nn.Module):
    def __init__(s,ci,co):
        super().__init__()
        s.conv=nn.Sequential(nn.Conv2d(ci,co,3,padding=1,bias=False),nn.BatchNorm2d(co),nn.GELU(),
            nn.Conv2d(co,co,3,padding=1,bias=False),nn.BatchNorm2d(co),nn.GELU())
        s.skip=nn.Conv2d(ci,co,1,bias=False) if ci!=co else None
    def forward(s,x):
        o=s.conv(x); return o+(s.skip(x) if s.skip else x)

class TCNBlock(nn.Module):
    """Dilated temporal convolutions via Conv3d"""
    def __init__(s, ch, n=3, drop=0.1):
        super().__init__()
        s.layers=nn.ModuleList()
        for i in range(n):
            d=2**i
            s.layers.append(nn.Sequential(
                nn.Conv3d(ch,ch,(3,1,1),padding=(d,0,0),dilation=(d,1,1),bias=False),
                nn.BatchNorm3d(ch), nn.GELU(), nn.Dropout3d(drop)))
    def forward(s,x):
        for l in s.layers: x=x+l(x)
        return x

class ConvLSTMCell(nn.Module):
    def __init__(s,ci,ch):
        super().__init__()
        s.ch=ch; s.gates=nn.Conv2d(ci+ch,4*ch,3,padding=1)
    def forward(s,x,h,c):
        g=s.gates(torch.cat([x,h],1)); i,f,o,g_=g.chunk(4,1)
        c=torch.sigmoid(f)*c+torch.sigmoid(i)*torch.tanh(g_)
        return torch.sigmoid(o)*torch.tanh(c), c

class ConvGRUCell(nn.Module):
    def __init__(s,ci,ch):
        super().__init__()
        s.ch=ch
        s.rz=nn.Conv2d(ci+ch,2*ch,3,padding=1)
        s.cand=nn.Conv2d(ci+ch,ch,3,padding=1)
    def forward(s,x,h):
        rz=torch.sigmoid(s.rz(torch.cat([x,h],1))); r,z=rz.chunk(2,1)
        h_=torch.tanh(s.cand(torch.cat([x,r*h],1)))
        return (1-z)*h+z*h_

class ChannelAttn(nn.Module):
    def __init__(s,ch,r=4):
        super().__init__()
        s.fc=nn.Sequential(nn.AdaptiveAvgPool2d(1),nn.Flatten(),
            nn.Linear(ch,ch//r),nn.GELU(),nn.Linear(ch//r,ch),nn.Sigmoid())
    def forward(s,x): return x*s.fc(x).unsqueeze(-1).unsqueeze(-1)

class V15Model(nn.Module):
    def __init__(s, ic=15, bc=48, hz=16, drop=0.20):
        super().__init__()
        s.hz=hz; c=bc; C4=c*4
        # Per-frame encoder
        s.fenc=nn.Sequential(nn.Conv2d(ic,48,3,padding=1,bias=False),nn.BatchNorm2d(48),nn.GELU())
        s.enc1=CB(48,c); s.enc2=CB(c,c*2); s.enc3=CB(c*2,C4)
        s.bot=CB(C4,C4)
        s.pool=nn.MaxPool2d(2)
        # TCN at bottleneck
        s.tcn=TCNBlock(C4, n=3, drop=drop*0.5)
        # BiLSTM
        hid=C4//2
        s.lstm_f=ConvLSTMCell(C4,hid); s.lstm_b=ConvLSTMCell(C4,hid)
        s.lstm_fuse=nn.Sequential(nn.Conv2d(hid*2,C4,1,bias=False),nn.BatchNorm2d(C4),nn.GELU())
        s.hid=hid
        # GRU (unidirectional)
        s.gru=ConvGRUCell(C4,C4)
        # Channel attention
        s.attn=ChannelAttn(C4)
        s.bn_drop=nn.Dropout2d(drop)
        # Decoder
        s.up3=nn.ConvTranspose2d(C4,c*2,2,stride=2); s.dec3=CB(c*4,c*2)
        s.up2=nn.ConvTranspose2d(c*2,c,2,stride=2); s.dec2=CB(c*2,c)
        s.up1=nn.ConvTranspose2d(c,c,2,stride=2); s.dec1=CB(c+48,c)
        s.head=nn.Sequential(nn.Conv2d(c,c,3,padding=1,bias=False),nn.BatchNorm2d(c),nn.GELU(),
            nn.Dropout2d(drop*0.5),nn.Conv2d(c,hz,1))

    def forward(s, x):
        B,T,C,Hh,Ww=x.shape
        base=x[:,-1,CPM25_CH,:,:].unsqueeze(1).expand(B,s.hz,Hh,Ww)
        # Per-frame encoder
        fr=s.fenc(x.reshape(B*T,C,Hh,Ww))
        sk1=fr.reshape(B,T,48,Hh,Ww)[:,-1]  # skip from last frame
        e1=s.enc1(fr); sk2=e1.reshape(B,T,*e1.shape[1:])[:,-1]
        e2=s.enc2(s.pool(e1)); sk3=e2.reshape(B,T,*e2.shape[1:])[:,-1]
        e3=s.enc3(s.pool(e2))
        bt=s.bot(s.pool(e3))  # (B*T, C4, bh, bw)
        C4=bt.shape[1]; bh=bt.shape[2]; bw=bt.shape[3]
        # TCN: (B, C4, T, bh, bw)
        seq=bt.reshape(B,T,C4,bh,bw).permute(0,2,1,3,4)
        seq=s.tcn(seq)  # temporal convolutions
        seq=seq.permute(0,2,1,3,4)  # (B,T,C4,bh,bw)
        # BiLSTM over T
        hf=torch.zeros(B,s.hid,bh,bw,device=x.device); cf=torch.zeros_like(hf)
        hb=torch.zeros(B,s.hid,bh,bw,device=x.device); cb=torch.zeros_like(hb)
        for t in range(T): hf,cf=s.lstm_f(seq[:,t],hf,cf)
        for t in range(T-1,-1,-1): hb,cb=s.lstm_b(seq[:,t],hb,cb)
        h=s.lstm_fuse(torch.cat([hf,hb],1))  # (B,C4,bh,bw)
        # GRU refinement (feed TCN output again)
        gh=h
        for t in range(T): gh=s.gru(seq[:,t],gh)
        # Channel attention
        h=s.attn(gh)
        h=s.bn_drop(h)
        # Decoder
        d=s.up3(h); d=F.interpolate(d,sk3.shape[2:],mode='bilinear',align_corners=False)
        d=s.dec3(torch.cat([d,sk3],1))
        d=s.up2(d); d=F.interpolate(d,sk2.shape[2:],mode='bilinear',align_corners=False)
        d=s.dec2(torch.cat([d,sk2],1))
        d=s.up1(d); d=F.interpolate(d,sk1.shape[2:],mode='bilinear',align_corners=False)
        d=s.dec1(torch.cat([d,sk1],1))
        return base+s.head(d)

model=V15Model(ic=IN_CH,bc=BASE_CH,hz=HORIZON,drop=DROPOUT).to(DEVICE)
np_=sum(p.numel() for p in model.parameters())/1e6
print(f'V15 TCN-BiLSTM-GRU-Attn: {np_:.2f}M params')
with torch.no_grad():
    o=model(torch.zeros(2,LOOKBACK,IN_CH,H,W,device=DEVICE))
    print(f'Output: {tuple(o.shape)} ✓')
del o; gc.collect(); torch.cuda.empty_cache()
swa_model=AveragedModel(model)
print('✅ Model ready')

V15 TCN-BiLSTM-GRU-Attn: 6.26M params
Output: (2, 16, 140, 124) ✓
✅ Model ready


In [5]:
# ═══ Loss: Lucifer exact (Huber+wSMAPE+Corr+Asym) ═══
class ACLoss(nn.Module):
    def __init__(s,hw=.45,sw=.25,cw=.15,aw=.15):
        super().__init__()
        s.hw=hw;s.sw=sw;s.cw=cw;s.aw=aw;s.hub=nn.SmoothL1Loss(beta=1.0)
    def forward(s,p,t):
        h=s.hub(p,t)
        d=torch.abs(p)+torch.abs(t)+1e-4; sm=2*torch.abs(p-t)/d
        mw=1+torch.relu(t); smape=(sm*mw).mean()/mw.mean()
        B,T_=p.shape[:2]; pf=p.reshape(B*T_,-1); tf=t.reshape(B*T_,-1)
        pc=pf-pf.mean(1,True); tc=tf-tf.mean(1,True)
        corr=1-((pc*tc).sum(1)/(torch.sqrt((pc**2).sum(1)*(tc**2).sum(1)+1e-8))).mean()
        hm=(t>.5).float(); asym=(torch.relu(t-p)*hm*(1+t)).mean()
        return s.hw*h+s.sw*smape+s.cw*corr+s.aw*asym

def comp_metrics(pr,tr,epm):
    ap=np.stack(pr);at=np.stack(tr);T_=ap.shape[1];ae=np.stack(epm)
    gs=[];es=[];ec=[]
    for t in range(T_):
        pt=ap[:,t].ravel();tt=at[:,t].ravel()
        d=.5*(np.abs(pt)+np.abs(tt))+1e-8; gs.append((np.abs(pt-tt)/d).mean())
        mk=ae[:,t].ravel().astype(bool); ep=pt[mk]; et=tt[mk]
        if len(ep)>=10:
            d2=.5*(np.abs(ep)+np.abs(et))+1e-8; es.append((np.abs(ep-et)/d2).mean())
            if ep.std()>1e-8 and et.std()>1e-8:
                c=np.corrcoef(ep,et)[0,1]
                if not np.isnan(c): ec.append(c)
    g=np.mean(gs); e=np.mean(es) if es else 1.; c=np.mean(ec) if ec else 0.
    return g,e,c,(1-g/2+(c+1)/2+1-e/2)/3

criterion=ACLoss()
print('Loss ✅')

Loss ✅


In [6]:
# ═══ Training ═══
optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=1e-4)
def lr_fn(ep):
    if ep<3: return (ep+1)/3
    if ep>=SWA_START: return 1e-4/LR
    pr=(ep-3)/max(1,SWA_START-3); return max(1e-6/LR,.5*(1+math.cos(math.pi*pr)))
sched=torch.optim.lr_scheduler.LambdaLR(optimizer,lr_fn)
scaler=torch.amp.GradScaler('cuda',enabled=USE_AMP)
best_comp=-1.; best_ep=-1; no_imp=0
SAVE=os.path.join(OUT_DIR,'best_v15.pt')

print(f'Training V15: {N_EPOCHS} epochs')
print(f'{"Ep":>3}|{"Loss":>7}|{"LR":>8}|{"gSMA":>6}|{"eCorr":>6}|{"eSMA":>6}|{"COMP":>6}| Note')
print('-'*80)
t0=time.time()
for epoch in range(N_EPOCHS):
    et0=time.time(); model.train(); el=[]
    for xb,yb in tl:
        xb,yb=xb.to(DEVICE),yb.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda',enabled=USE_AMP):
            loss=criterion(model(xb),yb)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer); nn.utils.clip_grad_norm_(model.parameters(),2.0)
        scaler.step(optimizer); scaler.update()
        el.append(loss.item())
    sched.step()
    if epoch>=SWA_START: swa_model.update_parameters(model)
    tl_=np.mean(el); lr=optimizer.param_groups[0]['lr']; dt=time.time()-et0
    do_val=(epoch%2==0)or(epoch>=N_EPOCHS-5)or(epoch<5)
    if do_val:
        if epoch>=SWA_START:
            torch.optim.swa_utils.update_bn(tl,swa_model,device=DEVICE)
            em=swa_model
        else: em=model
        em.eval(); pr_=[]; tr_=[]; epm_=[]
        with torch.no_grad():
            for xb,yb in vl:
                xb=xb.to(DEVICE)
                with torch.amp.autocast('cuda',enabled=USE_AMP): p=em(xb)
                pn=np.clip(norm.inv(p.float().cpu().numpy()),0,None)
                tn=norm.inv(yb.numpy())
                for i in range(pn.shape[0]): pr_.append(pn[i]); tr_.append(tn[i])
        for mi,t in val_idx[:len(pr_)]: epm_.append(mep[mi][t+LOOKBACK:t+WIN])
        g,e,c,comp=comp_metrics(pr_,tr_,epm_)
        sw='[SWA]' if epoch>=SWA_START else ''
        note=f'({dt:.0f}s){sw}'
        if comp>best_comp:
            best_comp=comp; best_ep=epoch+1; no_imp=0
            sd=swa_model.module.state_dict() if epoch>=SWA_START else model.state_dict()
            torch.save(sd,SAVE); note=f'✅BEST({dt:.0f}s){sw}'
        else: no_imp+=1
        print(f'{epoch+1:>3}|{tl_:.4f}|{lr:.2e}|{g:.4f}|{c:.4f}|{e:.4f}|{comp:.4f}|{note}')
        if no_imp>=PATIENCE and epoch<SWA_START: print('Early stop'); break
    else:
        print(f'{epoch+1:>3}|{tl_:.4f}|{lr:.2e}| skip |({dt:.0f}s)')
print(f'\n✅ Best ep={best_ep} comp={best_comp:.4f} ({(time.time()-t0)/60:.1f}min)')

Training V15: 30 epochs
 Ep|   Loss|      LR|  gSMA| eCorr|  eSMA|  COMP| Note
--------------------------------------------------------------------------------
  1|0.1459|3.33e-04|0.2436|0.8606|0.3052|0.8853|✅BEST(129s)
  2|0.1190|5.00e-04|0.2149|0.8778|0.2780|0.8975|✅BEST(101s)
  3|0.1118|5.00e-04|0.2124|0.8827|0.2626|0.9013|✅BEST(101s)
  4|0.1046|4.97e-04|0.2092|0.8795|0.2665|0.9006|(101s)
  5|0.1006|4.90e-04|0.2227|0.8836|0.2639|0.8995|(101s)
  6|0.0958|4.77e-04| skip |(101s)
  7|0.0918|4.60e-04|0.1944|0.8904|0.2409|0.9092|✅BEST(101s)
  8|0.0894|4.39e-04| skip |(101s)
  9|0.0864|4.14e-04|0.1976|0.8971|0.2379|0.9103|✅BEST(101s)
 10|0.0845|3.85e-04| skip |(101s)
 11|0.0809|3.54e-04|0.1920|0.8928|0.2389|0.9103|✅BEST(101s)
 12|0.0806|3.20e-04| skip |(101s)
 13|0.0776|2.86e-04|0.1884|0.8985|0.2346|0.9126|✅BEST(101s)
 14|0.0757|2.50e-04| skip |(101s)
 15|0.0747|2.14e-04|0.1867|0.9049|0.2256|0.9154|✅BEST(101s)
 16|0.0728|1.80e-04| skip |(101s)
 17|0.0717|1.46e-04|0.1853|0.9033|0.2249|0.915

In [7]:
# ═══ Inference: 4-flip TTA ═══
model.load_state_dict(torch.load(SAVE,map_location=DEVICE,weights_only=False))
model.eval()
print(f'Loaded best (ep={best_ep} comp={best_comp:.4f})')
print('Loading test...')
tf={}
for f in LIVE:
    a=np.load(os.path.join(TEST_DIR,f'{f}.npy')).astype(np.float32)
    if a.ndim==4 and a.shape[0]==1: a=a.squeeze(0)
    tf[f]=a; print(f'  {f}: {a.shape}')
te=eng_test(tf,norm); N=te.shape[0]
print(f'Test eng: {te.shape}')

IBS=4; flips=[(0,0),(1,0),(0,1),(1,1)]
tta=[]
for hf,vf in flips:
    bp=[]
    with torch.no_grad():
        for i in range(0,N,IBS):
            b=te[i:i+IBS].copy()
            if hf: b=b[:,:,:,:,::-1].copy()
            if vf: b=b[:,:,:,::-1,:].copy()
            with torch.amp.autocast('cuda',enabled=USE_AMP):
                p=model(torch.from_numpy(b).to(DEVICE))
            pn=p.float().cpu().numpy()
            if hf: pn=pn[:,:,:,::-1].copy()
            if vf: pn=pn[:,:,::-1,:].copy()
            bp.append(pn)
    tta.append(np.concatenate(bp))
pred=np.mean(tta,0)
pr=np.clip(norm.inv(pred),0,None).transpose(0,2,3,1).astype(np.float32)
print(f'Shape:{pr.shape} Range:[{pr.min():.1f},{pr.max():.1f}] Mean:{pr.mean():.1f}')
assert pr.shape==(218,140,124,16) and not np.isnan(pr).any()
np.save(os.path.join(OUT_DIR,'preds.npy'),pr)
print('✅ preds.npy saved!')

Loaded best (ep=30 comp=0.9162)
Loading test...
  cpm25: (218, 10, 140, 124)
  pblh: (218, 10, 140, 124)
  psfc: (218, 10, 140, 124)
  q2: (218, 10, 140, 124)
  rain: (218, 10, 140, 124)
  swdown: (218, 10, 140, 124)
  t2: (218, 10, 140, 124)
  u10: (218, 10, 140, 124)
  v10: (218, 10, 140, 124)
Test eng: (218, 10, 15, 140, 124)
Shape:(218, 140, 124, 16) Range:[0.0,1734.4] Mean:36.3
✅ preds.npy saved!
